# Setup

In [ ]:
import analysis as al
import matplotlib.gridspec as mgrid
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import struct
%matplotlib widget

In [ ]:
def add_label(text, ha='left', va='top', dx=0.03, dy=0.05, *, ax):
    x = dx if ha == 'left' else (1.0-dx)
    y = (1.0-dy) if va == 'top' else dy
    ax.text(x, y, text, ha=ha, va=va, transform=ax.transAxes, backgroundcolor=('w', 0.8))

In [ ]:
def smear_hex(X):
    assert len(X.shape) == 2
    kernel = np.array([
        [1, 1, 1, 1, 1],
        [1, 0, 0, 0, 1],
        [1, 1, 1, 1, 1],
    ])
    X = np.where(np.isfinite(X), X, 0)
    X = sp.signal.convolve2d(kernel, X, mode='valid')
    X = X[::2]
    out = []
    for x, Xi in enumerate(X):
        if x % 2 == 0:
            out.append(Xi[::4])
        else:
            out.append(Xi[2::4])
    return np.stack(out)

# Data

In [ ]:
def load(prefix, R):
    with open(f'{prefix}.meta.dat', 'rb') as f:
        meta_bytes = f.read()
    header = 'dddiiiI'
    h_size = struct.calcsize(header)
    (dt, KP, KE, NT, NX, NY, seed) = struct.unpack(header, meta_bytes[:h_size])
    print(f'Embedding shape: {NT=} {NX=} {NY=}')
    geom = np.frombuffer(meta_bytes[h_size:], dtype=np.uint8).reshape(NX, NY)
    Mx = np.fromfile(f'{prefix}.Mx.dat', dtype=np.float64).reshape(NX, NY)
    Ex = np.fromfile(f'{prefix}.Ex.dat', dtype=np.float64).reshape(NX, NY)
    tri_mask = np.zeros((NX, NY), dtype=np.uint8)
    tri_mask[::2,::2][geom[::2,::2] == 0xff] = 1
    pet_even_mask = np.zeros((NX, NY), dtype=np.uint8)
    pet_odd_mask = np.zeros((NX, NY), dtype=np.uint8)
    pet_even_mask[:,1::2][geom[:,1::2] == 0xff] = 1
    pet_odd_mask[1::2,:][geom[1::2,:] == 0xff] = 1
    pet_mask = pet_even_mask | pet_odd_mask
    E_vec = np.zeros((2, NX, NY))
    E_vec[0,0::2] = np.sqrt(3)/2
    E_vec[1,0::2,1::4] = -1/2
    E_vec[1,0::2,3::4] = 1/2
    E_vec[1,1::4,0::4] = 1
    E_vec[1,3::4,2::4] = 1
    mask = pet_mask | tri_mask
    return dict(
        geom=geom, tri_mask=tri_mask, pet_even_mask=pet_even_mask, pet_odd_mask=pet_odd_mask,
        pet_mask=pet_mask, mask=mask, NX=NX, NY=NY, NT=NT, Mx=Mx, Ex=Ex, E_vec=E_vec, R=R,
    )

In [ ]:
# load data
A = load('../data_ale/miao__0.7_0.0__96_48_48__0.05__diracstring_10_', 10)
B = load('../data_ale/miao__0.712_0.0__96_48_48__0.05__diracstring_10_', 10)

# Magnetization measurements

In [ ]:
A_Mx = A['Mx'] - 0.5
A_Mx[A['mask'] != 1] = float('nan')
B_Mx = B['Mx'] - 0.5
B_Mx[B['mask'] != 1] = float('nan')

In [ ]:
A_MTx = np.copy(A_Mx)
A_MPx = np.copy(A_Mx)
B_MTx = np.copy(B_Mx)
B_MPx = np.copy(B_Mx)
A_MTx[A['tri_mask'] != 1] = float('nan')
A_MPx[A['pet_mask'] != 1] = float('nan')
B_MTx[B['tri_mask'] != 1] = float('nan')
B_MPx[B['pet_mask'] != 1] = float('nan')

In [ ]:
A_MTx_smr = smear_hex(np.abs(A_MTx))
A_MPx_smr = smear_hex(np.abs(A_MPx))
B_MTx_smr = smear_hex(np.abs(B_MTx))
B_MPx_smr = smear_hex(np.abs(B_MPx))

In [ ]:
print(np.nanmax(np.abs(A_MTx)))
print(np.nanmax(np.abs(B_MTx)))
print(np.nanmax(np.abs(A_MPx)))
print(np.nanmax(np.abs(B_MPx)))

In [ ]:
cmap = plt.get_cmap('Grays').copy()
cmap.set_bad(color='k')
fig = plt.figure(figsize=(12,10), constrained_layout=True)
gs = mgrid.GridSpec(3, 4, figure=fig)
# fig, axes = plt.subplots(3,2,figsize=(12,10),tight_layout=True)
ax = fig.add_subplot(gs[0,:2])
ax.set_title('KP=0.7 |mag|')
ax.imshow(np.abs(A_MTx), interpolation='none', vmin=0, vmax=0.4, cmap=cmap)
ax = fig.add_subplot(gs[0,2:])
ax.set_title('KP=0.712 |mag|')
ax.imshow(np.abs(B_MTx), interpolation='none', vmin=0, vmax=0.06, cmap=cmap)
ax = fig.add_subplot(gs[1,:2])
ax.imshow(np.abs(A_MPx), interpolation='none', vmin=0, vmax=0.4, cmap=cmap)
ax = fig.add_subplot(gs[1,2:])
ax.imshow(np.abs(B_MPx), interpolation='none', vmin=0, vmax=0.06, cmap=cmap)
ax = fig.add_subplot(gs[2,0])
add_label('$|M_T|$', dx=0.05, ax=ax)
ax.imshow(A_MTx_smr)
ax = fig.add_subplot(gs[2,1])
add_label('$|M_P|$', dx=0.05, ax=ax)
ax.imshow(A_MPx_smr)
ax = fig.add_subplot(gs[2,2])
add_label('$|M_T|$', dx=0.05, ax=ax)
ax.imshow(B_MTx_smr)
ax = fig.add_subplot(gs[2,3])
add_label('$|M_P|$', dx=0.05, ax=ax)
ax.imshow(B_MPx_smr)
fig.savefig('mag_2d.pdf')
plt.show()

# Flux from magnetization

In [ ]:
A_MTx_above = np.roll(A_MTx, 1, axis=0)
A_MTx_below = np.roll(A_MTx, -1, axis=0)
A_MTx_below[A['NX']//2+1, (A['NY']-4*A['R'])//2:(A['NY']+4*A['R'])//2] -= 0.5
A_Ex_check = (
    (A_MPx - 0.5 - A_MTx_above) % 2 - 1
    + (A_MTx_below - A_MPx + 0.5) % 2 - 1
)
A_Ex_check[A['pet_odd_mask'] != 1] = float('nan')
# A_Ex_check[A['NX']//2+1] = 0 # mask Dirac string
B_Ex_check = (
    (B_MPx - 0.5 - np.roll(B_MTx, 1, axis=0)) % 2 - 1
    + (np.roll(B_MTx, -1, axis=0) - B_MPx + 0.5) % 2 - 1
)
B_Ex_check[B['pet_odd_mask'] != 1] = float('nan')
B_Ex_check[B['NX']//2+1] = 0 # mask Dirac string

In [ ]:
cmap = plt.get_cmap('Grays_r').copy()
cmap.set_bad(color='k')
fig, axes = plt.subplots(1,2, figsize=(12, 6))
axes[0].imshow(np.abs(A_Ex_check), cmap=cmap)
axes[1].imshow(np.abs(B_Ex_check), cmap=cmap)
plt.show()

# Flux measurements

In [ ]:
A_Ex = A['Ex']
A_Ex[A['pet_mask'] != 1] = float('nan')
B_Ex = B['Ex']
B_Ex[B['pet_mask'] != 1] = float('nan')
A_Ev = A_Ex * A['E_vec']
A_Ev_smr = np.stack([smear_hex(A_Ev[0]), smear_hex(A_Ev[1])])
B_Ev = B_Ex * B['E_vec']
B_Ev_smr = np.stack([smear_hex(B_Ev[0]), smear_hex(B_Ev[1])])

In [ ]:
print(np.nanmax(A_Ex), np.nanmin(A_Ex))
print(np.nanmax(B_Ex), np.nanmin(B_Ex))

In [ ]:
cmap = plt.get_cmap('Grays_r').copy()
cmap.set_bad(color='k')
cmap2 = plt.get_cmap('PuOr').copy()
cmap2.set_bad(color='w')
fig = plt.figure(figsize=(12,6), constrained_layout=True)
gs = mgrid.GridSpec(2, 4, figure=fig, height_ratios=[0.6, 0.4])
# fig, axes = plt.subplots(4,4,figsize=(12,10),tight_layout=True)
ax = fig.add_subplot(gs[0,:2])
ax.set_title('KP=0.7 flux')
ax.imshow(np.abs(A_Ex), interpolation='none', vmin=0.0, vmax=1e-2, cmap=cmap)
ax = fig.add_subplot(gs[0,2:])
ax.set_title('KP=0.712 flux')
ax.imshow(np.abs(B_Ex), interpolation='none', vmin=0.0, vmax=1e-2, cmap=cmap)
# axes[1,0].imshow(A_Ex*A['E_vec'], interpolation='none', vmin=-1e-2, vmax=1e-2, cmap=cmap2)
# axes[1,1].imshow(B_Ex*B['E_vec'], interpolation='none', vmin=-1e-2, vmax=1e-2, cmap=cmap2)
ax = fig.add_subplot(gs[1,0])
add_label(r'$E_{\rm vert}$', dx=0.05, ax=ax)
ax.imshow(A_Ev_smr[0], interpolation='none', vmin=-1e-1, vmax=1e-1, cmap=cmap2)
ax = fig.add_subplot(gs[1,1])
add_label(r'$E_{\rm horiz}$', dx=0.05, ax=ax)
ax.imshow(A_Ev_smr[1], interpolation='none', vmin=-1e-1, vmax=1e-1, cmap=cmap2)
ax = fig.add_subplot(gs[1,2])
add_label(r'$E_{\rm vert}$', dx=0.05, ax=ax)
ax.imshow(B_Ev_smr[0], interpolation='none', vmin=-1e-1, vmax=1e-1, cmap=cmap2)
ax = fig.add_subplot(gs[1,3])
add_label(r'$E_{\rm horiz}$', dx=0.05, ax=ax)
ax.imshow(B_Ev_smr[1], interpolation='none', vmin=-1e-1, vmax=1e-1, cmap=cmap2)
fig.savefig('flux_2d.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1)
for x, ls in zip([12, 24, 36], ['-', '--', '-.']):
    ax.plot(A_Ev_smr[1][:,x], color='tab:blue', label=f'KP=0.7 [x={x}]', ls=ls)
    ax.plot(B_Ev_smr[1][:,x], color='tab:red', label=f'KP=0.712 [x={x}]', ls=ls)
ax.legend()
ax.set_xlabel('Transverse distance $y$')
ax.set_title('Flux profile')
fig.savefig('flux_transverse.pdf')
plt.show()